# Variational Autoencoder for Out-of-Distribution Rejection

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 7: Autoencoders and Variational Autoencoders**

The first line of defence in the deployed system. A convolutional VAE is trained
**only on chest radiographs**, so it reconstructs them well and everything else
badly. Reconstruction error then separates in-distribution from out-of-distribution
inputs, and the classifier never sees an image it has no business judging.

This notebook produces the OOD threshold used in production, chosen at a fixed
false-positive rate rather than by eyeballing a histogram.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. Model and the ELBO

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent=128, base=32):
        super().__init__(); self.latent, self._base = latent, base
        def enc(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True))
        def dec(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.encoder = nn.Sequential(enc(1, base), enc(base, base*2), enc(base*2, base*4), enc(base*4, base*8))
        self.flat = base*8*8*8
        self.fc_mu, self.fc_logvar = nn.Linear(self.flat, latent), nn.Linear(self.flat, latent)
        self.fc_dec = nn.Linear(latent, self.flat)
        self.decoder = nn.Sequential(dec(base*8, base*4), dec(base*4, base*2), dec(base*2, base),
                                     nn.ConvTranspose2d(base, 1, 4, 2, 1), nn.Sigmoid())

    def encode(self, x):
        h = self.encoder(x).flatten(1); return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        # Deterministic at eval so OOD scores are reproducible run to run.
        if not self.training: return mu
        return mu + torch.exp(0.5*logvar) * torch.randn_like(mu)

    def decode(self, z): return self.decoder(self.fc_dec(z).view(-1, self._base*8, 8, 8))

    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    """ELBO = reconstruction + beta * KL.

    beta > 1 (Higgins et al.) trades fidelity for a more disentangled latent
    space. For OOD detection we want fidelity, so beta stays near 1 — a heavily
    disentangled VAE reconstructs everything mediocrely and the score separates
    less well.
    """
    rec = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return rec + beta*kl, rec, kl

v = ConvVAE(); r, mu, lv = v(torch.rand(2,1,128,128))
print("reconstruction", tuple(r.shape), "| latent", tuple(mu.shape))

## 2. Choosing the threshold

At a fixed false-positive rate on real radiographs — never by eye.

In [ ]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def reconstruction_errors(model, loader):
    model.eval(); errs = []
    for x, *_ in loader:
        x = x.to(DEVICE); recon, _, _ = model(x)
        errs.append(F.mse_loss(recon, x, reduction="none").flatten(1).mean(1).cpu().numpy())
    return np.concatenate(errs)

def choose_threshold(in_dist_errors, ood_errors, target_fpr=0.01):
    """Threshold at `target_fpr` on IN-DISTRIBUTION data.

    Rejecting 1% of genuine radiographs is an acceptable cost; rejecting 10%
    would make the system unusable. The FPR budget is the design decision, so
    it is set explicitly rather than implied by a round-number threshold.
    """
    tau = float(np.quantile(in_dist_errors, 1 - target_fpr))
    tpr = float((ood_errors > tau).mean())
    y = np.r_[np.zeros(len(in_dist_errors)), np.ones(len(ood_errors))]
    s = np.r_[in_dist_errors, ood_errors]
    print(f"threshold tau            : {tau:.6f}")
    print(f"radiographs wrongly rejected: {target_fpr:.1%}")
    print(f"non-radiographs caught     : {tpr:.1%}")
    print(f"gate AUROC                 : {roc_auc_score(y, s):.4f}")
    print(f"\nSet OOD_THRESHOLD={tau:.6f} in the API environment.")
    return tau

print("Threshold selection ready.")
print("For the OOD set use CIFAR-10 or any natural-image corpus — the point is")
print("that they are obviously not radiographs, which is the case the gate must")
print("catch. A harder OOD set (e.g. abdominal X-rays) is a stronger test and is")
print("reported separately as a limitation.")

## 3. Latent space — the generative side of the syllabus

In [ ]:
@torch.no_grad()
def latent_traversal(model, x, dim=0, span=3.0, steps=7):
    """Walk one latent dimension and decode, showing what it encodes."""
    model.eval()
    mu, _ = model.encode(x[:1].to(DEVICE))
    fig, axes = plt.subplots(1, steps, figsize=(steps*1.4, 1.7))
    for i, v in enumerate(np.linspace(-span, span, steps)):
        z = mu.clone(); z[0, dim] = v
        axes[i].imshow(model.decode(z)[0,0].cpu().numpy(), cmap="gray")
        axes[i].axis("off"); axes[i].set_title(f"{v:+.1f}", fontsize=7)
    plt.suptitle(f"Latent dimension {dim}", fontsize=9); plt.tight_layout(); plt.show()

print("latent_traversal ready — run after training to show what the VAE learned.")

---

### References for this notebook

- Kingma, D. P. & Welling, M. (2014). Auto-encoding variational Bayes. *ICLR*.
- Burgess, C. P. et al. (2018). Understanding disentangling in beta-VAE. arXiv:1804.03599.
- Kingma, D. P. & Dhariwal, P. (2018). Glow: generative flow with invertible 1x1 convolutions. *NeurIPS*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
